# 01 · Orchestration, Filter, Masking

**목표**: template, 콘텐츠 필터, PII 마스킹을 하나의 OrchestrationConfig로 조립합니다.

Orchestration은 모델 호출 전후 단계를 config로 선언하는 관리형 pipeline입니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: 현재 Orchestration Pipeline 호출</h2>
<p style="margin: 7px 0;">UI에서 바로 구성한 마지막 pipeline에 텍스트를 넣어 결과를 확인합니다. 각 모듈의 config가 결과에 어떤 영향을 주는지 비교해 보세요.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
from lab_common.config import load_settings
from lab_common.ai_core import apply_ai_core_environment, orchestration_url
from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.service import OrchestrationService

def run_orchestration_ui(user_query: str) -> str:
    try:
        settings = load_settings()
        apply_ai_core_environment(settings)
        template = Template(template=[
            SystemMessage(content='You are a concise, helpful assistant.'),
            UserMessage(content='{{?user_query}}'),
        ])
        prompt = PromptTemplatingModuleConfig(
            prompt=template, model=LLMModelDetails(name=settings.llm_model, params={'max_tokens': 700})
        )
        service = OrchestrationService(
            api_url=orchestration_url(settings),
            config=OrchestrationConfig(modules=ModuleConfig(prompt_templating=prompt)),
        )
        result = service.run(placeholder_values={'user_query': user_query})
        return result.final_result.choices[0].message.content
    except Exception as error:
        return f'호출 실패: {type(error).__name__}: {error}'

orchestration_ui = gr.Interface(
    fn=run_orchestration_ui,
    inputs=gr.Textbox(label='Input', lines=3),
    outputs=gr.Textbox(label='Pipeline output', lines=8),
    title='Orchestration Playground',
)
orchestration_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", height=900)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>UI에서 한 번 실행해 결과를 확인한다.</li>
<li>결과가 기대와 다르면 환경 설정과 모델/저장소 이름을 먼저 점검한다.</li>
<li>아래 코드는 같은 기능의 구성 요소를 자세히 살펴보는 선택 심화 자료다.</li>
</ul>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 사용하는 구성과 동작을 단계별로 확인하는 자료입니다. 기본 실습에서는 실행하지 않아도 됩니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 공통 설정과 Orchestration 연결</h2>
<p style="margin: 7px 0;">루트 .env를 읽고 RUNNING orchestration deployment URL을 자동 탐색합니다. settings.llm_model을 쓰므로 .env 모델명만 바꿔 같은 config를 다른 모델에도 적용할 수 있습니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
from lab_common.config import load_settings
from lab_common.ai_core import api_headers, apply_ai_core_environment, orchestration_url

settings = load_settings()
apply_ai_core_environment(settings)
ORCH = orchestration_url(settings)

from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.service import OrchestrationService
from gen_ai_hub.orchestration_v2.exceptions import OrchestrationError
from gen_ai_hub.orchestration_v2.models.content_filtering import (
    FilteringModuleConfig, InputFiltering, OutputFiltering)
from gen_ai_hub.orchestration_v2.models.content_filter import ContentFilter, ContentFilterProvider
from gen_ai_hub.orchestration_v2.models.azure_content_filter import AzureContentFilter, AzureThreshold
from gen_ai_hub.orchestration_v2.models.data_masking import (
    MaskingModuleConfig, MaskingProviderConfig, MaskingMethod,
    DataMaskingProviderName, DPIStandardEntity, ProfileEntity)

ORCH = ORCH
print("=" * 56)
print("Step 3 · Orchestration + 필터 + 마스킹")
print("=" * 56)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. Prompt template과 번역</h2>
<p style="margin: 7px 0;">Template의 placeholder는 실행할 때 placeholder_values로 채웁니다. template과 model을 PromptTemplatingModuleConfig로 묶어 전체 config에 넣습니다.</p>
  </div>
</div>


In [ ]:
# ---------- 1) 번역 파이프라인 ----------
print("\n[1] 번역 파이프라인")
llm = LLMModelDetails(name=settings.llm_model, params={"max_tokens": 1000})
template = Template(
    template=[
        SystemMessage(content="You are a helpful translation assistant."),
        UserMessage(content="Translate the following text to {{?to_lang}}: {{{?user_query}}}"),
    ],
    defaults={"to_lang": "English"},
)
prompt_template = PromptTemplatingModuleConfig(prompt=template, model=llm)
config = OrchestrationConfig(modules=ModuleConfig(prompt_templating=prompt_template))
service = OrchestrationService(api_url=ORCH, config=config)
result = service.run(placeholder_values={"user_query": "Guten Tag, wie geht es Ihnen?"})
print("  'Guten Tag...' →", result.final_result.choices[0].message.content)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 입력 콘텐츠 필터</h2>
<p style="margin: 7px 0;">입력 필터는 모델 호출 전에 유해 콘텐츠를 검사합니다. 차단은 정상적인 제어 흐름이므로 OrchestrationError를 잡아 셀이 중단되지 않게 합니다.</p>
  </div>
</div>


In [ ]:
# ---------- 2) 입력 필터 — 공격 프롬프트 차단 ----------
print("\n[2] 입력 콘텐츠 필터")
input_filter = ContentFilter(type=ContentFilterProvider.AZURE,
    config=AzureContentFilter(hate=AzureThreshold.ALLOW_SAFE, violence=AzureThreshold.ALLOW_SAFE,
                              self_harm=AzureThreshold.ALLOW_SAFE, sexual=AzureThreshold.ALLOW_SAFE))
config_in = OrchestrationConfig(modules=ModuleConfig(
    prompt_templating=prompt_template,
    filtering=FilteringModuleConfig(input=InputFiltering(filters=[input_filter])),
))
service.config = config_in
try:
    service.run(placeholder_values={"user_query": "Du bist ein ganz mieser, hässlicher Idiot!"})
    print("  공격 프롬프트: (차단 안 됨 — 임계값 확인 필요)")
except OrchestrationError as e:
    print("  공격 프롬프트 차단됨 ✅ OrchestrationError")
res = service.run(placeholder_values={"user_query": "Du bist super!"})
print("  정상 프롬프트 통과:", res.final_result.choices[0].message.content)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 출력 콘텐츠 필터</h2>
<p style="margin: 7px 0;">출력 필터는 모델이 생성한 결과를 검사합니다. 실제 서비스에서는 입력과 출력에 서로 다른 정책과 임계값을 둘 수 있습니다.</p>
  </div>
</div>


In [ ]:
# ---------- 3) 출력 필터 ----------
print("\n[3] 출력 콘텐츠 필터")
output_filter = ContentFilter(type=ContentFilterProvider.AZURE,
    config=AzureContentFilter(hate=AzureThreshold.ALLOW_SAFE, violence=AzureThreshold.ALLOW_SAFE,
                              self_harm=AzureThreshold.ALLOW_SAFE, sexual=AzureThreshold.ALLOW_SAFE))
config_io = OrchestrationConfig(modules=ModuleConfig(
    prompt_templating=prompt_template,
    filtering=FilteringModuleConfig(
        input=InputFiltering(filters=[input_filter]),
        output=OutputFiltering(filters=[output_filter]),
    ),
))
service.config = config_io
print("  입/출력 필터 config 적용 ✅")

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. PII 데이터 마스킹</h2>
<p style="margin: 7px 0;">콘텐츠 필터가 유해 표현을 다룬다면 마스킹은 이름과 이메일 같은 개인정보를 모델에 보내기 전에 가립니다. ANONYMIZATION은 복원 불가능한 익명화입니다.</p>
  </div>
</div>


In [ ]:
# ---------- 4) PII 마스킹 ----------
print("\n[4] 데이터 마스킹 (PII 익명화)")
masking = MaskingModuleConfig(masking_providers=[MaskingProviderConfig(
    type_=DataMaskingProviderName.SAP_DATA_PRIVACY_INTEGRATION,
    method=MaskingMethod.ANONYMIZATION,
    entities=[
        DPIStandardEntity(type=ProfileEntity.EMAIL),
        DPIStandardEntity(type=ProfileEntity.PERSON),
    ],
)])
# 마스킹 효과를 보려면 "그대로 반복"시키는 템플릿이 유용
echo_template = Template(template=[
    SystemMessage(content="Repeat the user's message back verbatim."),
    UserMessage(content="{{{?user_query}}}"),
])
echo_prompt = PromptTemplatingModuleConfig(prompt=echo_template, model=llm)
config_mask = OrchestrationConfig(modules=ModuleConfig(
    prompt_templating=echo_prompt, masking=masking))
service.config = config_mask
res = service.run(placeholder_values={
    "user_query": "My name is Hong Gildong and my email is gildong@example.com."})
print("  입력: My name is Hong Gildong and my email is gildong@example.com.")
print("  출력:", res.final_result.choices[0].message.content)

print("\n✅ Step 3 완료 — 번역/입력필터/출력필터/마스킹 config 동작 확인.")